# Day 33 — Practice Questions
### Drawing the Best Line

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A needs no computer.** The numbers are chosen so the arithmetic is clean. Work it
out in the margin, *then* run the cell to check.

The three questions to keep asking:

- Is a straight line even the **right shape** for this data?
- Is the number I am predicting **inside the range** I actually saw?
- What do the errors look like — random, or a **pattern**?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("ready")

---
# Part A — Work it out

Eight questions. No computer needed for most of them.

**The two formulas:**

```
slope      =  Σ (x − x̄)(y − ȳ)  ÷  Σ (x − x̄)²
intercept  =  ȳ  −  slope × x̄
```

### A1 · Find the best line by hand

Five students: practice problems solved per week, and their quiz score out of 50.

| problems | 2 | 4 | 6 | 8 | 10 |
|---|---|---|---|---|---|
| **score** | 19 | 27 | 33 | 37 | 44 |

Work out the slope and the intercept. Build the table of `x − x̄`, `y − ȳ`, their product,
and `(x − x̄)²` first.

In [ ]:
x = np.array([2., 4, 6, 8, 10])
y = np.array([19., 27, 33, 37, 44])

t = pd.DataFrame({"x": x, "y": y, "x − x̄": x - x.mean(), "y − ȳ": y - y.mean()})
t["product"] = t["x − x̄"] * t["y − ȳ"]
t["(x − x̄)²"] = t["x − x̄"] ** 2
print("x̄ =", x.mean(), "   ȳ =", y.mean())
t

### A2 · Total squared error

Using your line from A1, work out the error on each student and the total squared error.

In [ ]:
# fill in your line here, then run
pred = 3 * x + 14
err = y - pred

print("the line says :", pred)
print("they got      :", y)
print("errors        :", err)
print()
print("errors added  :", err.sum())
print("squares added :", (err ** 2).sum())

### A3 · Beat it, if you can

Try `slope = 4, intercept = 8` and `slope = 2, intercept = 20`. Both pass through the
average point (6, 32). Do either of them beat a total squared error of 4?

In [ ]:
def total_squared_error(slope, intercept):
    return float(((y - (slope * x + intercept)) ** 2).sum())

for sl, ic in [(2, 20), (3, 14), (4, 8)]:
    print(f"score = {sl} × problems + {ic:<3}  ->  total squared error = {total_squared_error(sl, ic):>5.0f}")

### A4 · What does the intercept mean here?

Your line says `score = 3 × problems + 14`.

- What does 14 mean in plain English?
- Should you quote it to anybody?

### A5 · R² by hand

Using total squared error 4 from A2, work out R² — no `sklearn`.

*Reminder:* R² = 1 − (your line's total squared error ÷ the flat average's total squared error)

In [ ]:
flat = np.full_like(y, y.mean())
print("if we guessed", y.mean(), "every time, the squared errors would be:")
print("   ", (y - flat) ** 2)
print("    total:", ((y - flat) ** 2).sum())

### A6 · One step of gradient descent

Start at slope 0 and intercept 0 on the A1 data. With a learning rate of 0.01, work out
the slope and intercept after **one** step.

*Reminder:*
```
slope nudge      =  (2/n) × Σ (error × x)        where error = line − actual
intercept nudge  =  (2/n) × Σ (error)
new value        =  old value − rate × nudge
```

In [ ]:
e = (0 * x + 0) - y          # line minus actual, starting from a flat line at zero
print("errors (line − actual):", e)
print("Σ(error × x) =", (e * x).sum())
print("Σ(error)     =", e.sum())

### A7 · Which of these should NOT be modelled with a straight line?

For each, say yes or no and why.

| | The situation |
|---|---|
| a | Flat area in square feet → monthly rent |
| b | Hours of study → exam mark, where marks are capped at 100 and most students already score 85+ |
| c | Age of a car → its resale price |
| d | Number of people in a restaurant → the noise level |

### A8 · Two models, one dataset

Two students fit a line to the same 200 rows.

| | R² on the rows it studied | R² on hidden rows |
|---|---|---|
| **Priya** | 0.71 | 0.69 |
| **Arjun** | 0.98 | 0.34 |

What happened in each case, and whose model would you ship?

---
# Part B — Find the mistake

Four short programs. Each one runs cleanly and reports something misleading.

### B1 · "R² is 0.83, so we are done"

In [ ]:
rng = np.random.default_rng(401)
n = 90
bx = np.linspace(1, 20, n) + rng.normal(0, 0.2, n)      # weeks on the training programme
by = (12*np.log(bx) + 18 + rng.normal(0, 3, n)).round(1)  # fitness score

model = LinearRegression().fit(bx.reshape(-1, 1), by)
print("slope :", round(model.coef_[0], 3))
print("R²    :", round(model.score(bx.reshape(-1, 1), by), 4))

### B2 · The flat-pricing model

In [ ]:
rng = np.random.default_rng(402)
area = rng.uniform(400, 1400, 70).round(0)
rent = (3000 + 21*area + rng.normal(0, 2200, 70)).round(0)

model = LinearRegression().fit(area.reshape(-1, 1), rent)

print(f"rent = {model.coef_[0]:.1f} × area + {model.intercept_:.0f}")
print()
print(f"a 5,000 sqft penthouse would rent for ₹{model.predict([[5000]])[0]:,.0f}")
print(f"an empty plot (0 sqft) would rent for ₹{model.intercept_:,.0f}")

### B3 · "Our model scores 0.88"

In [ ]:
X_train, X_hidden, y_train, y_hidden = train_test_split(
    area.reshape(-1, 1), rent, test_size=0.3, random_state=0)

model = LinearRegression().fit(X_train, y_train)

print("score:", round(model.score(X_train, y_train), 4))     # <-- look here

### B4 · One typo, and the model falls over

In [ ]:
rng = np.random.default_rng(403)
hours = np.append(rng.uniform(1, 10, 40).round(1), 9.8)
score = np.append((30 + 5*hours[:40] + rng.normal(0, 3, 40)).round(0), 15.0)
#                                   the last student studied 9.8 hours and "scored 15"  ^^^^

model = LinearRegression().fit(hours.reshape(-1, 1), score)
print("slope :", round(model.coef_[0], 3))
print("R²    :", round(model.score(hours.reshape(-1, 1), score), 4))

---
# Part C — Write the code

Five tasks on one dataset.

In [ ]:
# ---------------------------------------------------------------
# 120 delivery riders. Run this once — C1 to C5 all use `riders`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3301)
N = 120

experience = rng.uniform(1, 48, N).round(0)                        # months on the job
per_day = (6 + 0.22*experience + rng.normal(0, 1.5, N)).clip(2, None).round(1)

riders = pd.DataFrame({"experience_months": experience,
                       "deliveries_per_day": per_day})
riders.head()

### C1 · Look before you fit

Draw the scatter plot. Before fitting anything, write one sentence: does a straight line
look like the right shape here?

In [ ]:
# your code here

### C2 · Fit it, and say what the two numbers mean

Fit a `LinearRegression` on all 120 riders. Print the slope and intercept, then write
**one sentence each** saying what they mean — in words a delivery manager would
understand.

In [ ]:
# your code here

### C3 · Score it honestly

Split the riders, fit on the training rows, and report:

- R² on the rows it studied, and on the hidden rows
- the average miss, in deliveries, on the hidden rows
- how that compares with just guessing the average for everybody

In [ ]:
# your code here

### C4 · Check the errors

Plot the errors on the hidden rows against experience. Does the line fit, or is it the
wrong shape? Back your answer with the average error in four slices of the data.

In [ ]:
# your code here

### C5 · Answer three questions from management

Using your model, and refusing where you should:

1. A rider has been with us **18 months**. How many deliveries a day should we expect?
2. If we keep a rider an extra **year**, what do we gain?
3. What about a rider with **10 years** (120 months) of experience?

In [ ]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build least squares from scratch

Write **two** functions that find the best line, using no scikit-learn at all.

1. `fit_formula(x, y)` — uses the formula from the lecture. Returns `(slope, intercept)`.
2. `fit_descent(x, y, rate, steps)` — uses gradient descent. Returns `(slope, intercept)`.

Then prove all three agree — your two functions and `LinearRegression` — on the five
students from the lecture.

**Marks are for:**
- both functions working on any `x` and `y`, not just this dataset
- `fit_descent` reporting its progress so a reader can see it converging
- a clear demonstration that all three land in the same place

In [ ]:
hours = np.array([1., 2, 3, 4, 5])
marks = np.array([34., 44, 47, 58, 67])

In [ ]:
# your two functions here

### D2 · An R² of 0.93. Prove the claim is worthless.

A startup's marketing head sends this to the board:

> *"We fitted a regression of revenue against ad spend across our last 36 months. **R² is
> 0.93** — the relationship could not be clearer. Every ₹1,000 we put into advertising
> returns **₹7,735** in revenue. I am therefore requesting a budget of **₹500,000** for
> next month, which the model projects will deliver **₹3.9 million** in revenue."*

The regression is correctly fitted. There is no bug in the code.

**Your tasks:**

1. Run it and confirm the numbers.
2. Find **two separate reasons** the conclusion does not follow. They are different kinds
   of mistake.
3. Prove each one with evidence from the data.
4. Write three or four sentences you could send back to the board.

*Hints, in increasing order of giving it away: look at the ratio of the two columns · look
at what else is in the table · look at the range.*

In [ ]:
rng = np.random.default_rng(5150)
n = 36                                                  # three years of months

month   = np.arange(n)
revenue = (420000 + 9500*month                          # the company is growing
           + 95000*np.sin(2*np.pi*(month % 12)/12 - 0.8)   # and it is seasonal
           + rng.normal(0, 38000, n)).round(0)

# the marketing team sets each month's budget from what they expect to sell
ad_spend = (0.12*revenue + rng.normal(0, 4000, n)).round(0)

books = pd.DataFrame({"month": month, "ad_spend": ad_spend, "revenue": revenue})
books.head(8)

In [ ]:
claim = LinearRegression().fit(books[["ad_spend"]], books["revenue"])

print(f"R²                      : {claim.score(books[['ad_spend']], books['revenue']):.4f}")
print(f"every ₹1 of ads returns : ₹{claim.coef_[0]:.3f}")
print(f"so ₹1,000 returns       : ₹{claim.coef_[0]*1000:,.0f}")
print(f"spend ₹500,000 and get  : ₹{claim.predict([[500000]])[0]:,.0f}")

In [ ]:
# your investigation here

---
## Before you hand it in

- Did you **draw the scatter plot** before fitting anything?
- Can you finish the sentence *"one more unit of X is worth ___ of Y"*?
- Is `x = 0` inside your data? If not, you may not quote the intercept.
- Is every number you predicted **inside the range** you actually observed?
- Did you plot the **errors** against the input, and check for a pattern?
- Did you report R² **against a baseline**, not on its own?
- Did you say "moves with" rather than "causes"?

Four numbers worth carrying away:

- The best line on the five students had total squared error **14**. The exactly
  upside-down line scored **2,574** — and both had errors that added to **zero**.
- The formula and gradient descent both landed on slope **8.000000**. Same bowl, one
  bottom.
- One mistyped row out of 41 cut R² from **0.949 to 0.500**. Least squares squares its
  errors, so one bad point can outweigh forty good ones.
- An R² of **0.93** was produced by a growing company and a percentage-based budget, with
  no causation anywhere in it.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*